# Metronome Compass Calibration

A Python port of RNGReporter's HGSS **Seed to Time** verification panel, built for
gathering Metronome-compass calibration data.

Give it a target datetime + delay and a search window, and it enumerates every candidate
seed nearby.  For each seed it reports:

- **Roamer relocation** — where Raikou / Entei / Latios(Latias) move to when the save is
  reloaded (given where they are now).
- **Elm phone-call sequence** — the `P`/`E`/`K` calls you can read off in-game.

Two sections identify the seed two ways: **Section A** from roamer routes + Elm calls
(`a_seed`), **Section B** from the Metronome battle (`b_seed`).  All logic lives in
`utils/calibration_tools.py`.

## Keyboard fixup

The `2` and `w` keys on my keyboard are flaky, so every `input()` prompt below accepts
`\T` for `2` and `\V` for `w` (substituted before the value is used).  It's applied
automatically at each interactive step — ipykernel resets `input` once per cell, so the
library re-installs the fixup at every prompt.  To add pairs, edit `INPUT_SUBS` in
`utils/calibration_tools.py`.


In [2]:
%load_ext autoreload
%autoreload 2
import datetime as dt
from utils.calibration_tools import (
    # Section A -- roamer routes + Elm calls
    generate_roamer_candidates_near,
    print_roamer_candidates,
    identify_seed,
    # Section B -- Metronome-compass battle
    generate_candidates_near,
    print_candidates,
    narrow_candidates,
    prompt_magikarp,
    # Persist a run
    save_compass_run,
    # Timer calibration math
    calibrate_timer,
)


## Section A — Roamer + Elm identification  (→ `a_seed`)

**Configure** your target datetime/delay, the search window, and each roamer's **current**
route (where it is *now*, before the reset) plus whether it's still roaming.  Use `0` for a
current route you don't know or care about.  Variables are `a_`-prefixed so they won't clash
with Section B.

**Identify** (after loading the save, read the roamer map and Elm phone):

1. **Roamer routes** — one number per *roaming* legendary in **R E L** order (e.g. `38 42 11`);
   `.` leaves a roamer unconstrained.  Only roamers marked `present` are expected.
2. If more than one candidate matches, **Elm calls** — type `P`/`E`/`K` as you hear each
   call (matched as a substring, since RNG may advance first); other characters are ignored.
   Type `M` to pick a candidate by number instead.
3. The single surviving row is saved to **`a_seed`** (integer seed is `a_seed["seed"]`).


In [94]:
# --- Section A: roamer / Elm target + current roamer state ---
a_target_time    = dt.datetime(2025, 7, 24, 14, 45, 55)
a_target_delay   = 681
a_seconds_window = 1        # +/- X seconds
a_delay_window   = 60       # +/- Y delays
a_match_parity   = True     # only delays with target_delay's even/odd parity
a_display_limit  = 40       # rows to print (None = all)

# Each roamer's CURRENT route (before the reset) and whether it30 is still roaming.
a_prev_routes = {"r": 43, "e": 45, "l": 6}
a_present     = {"r": True, "e": True, "l": True}

a_candidates = generate_roamer_candidates_near(
    a_target_time, a_target_delay, a_seconds_window, a_delay_window,
    prev_routes=a_prev_routes, present=a_present, match_parity=a_match_parity,
)

# Interactively pin down the seed: roamer routes -> Elm calls -> (M) manual pick.
a_seed = identify_seed(a_candidates, a_present, display_limit=a_display_limit)
# GOALS: EKP, KPEK, PEKK, EKKP

Observed roamer routes (R E L, space-separated, . = any):  32 33 17



Observed R=32 E=33 L=17  ->  3 / 183 candidate(s) match

3 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0B0E02C6  2025-07-24 14:45:54     685    +4   -1   32  33  17   3  EKEKEPKPPPEPEEE
  0x0C0E02C6  2025-07-24 14:45:55     685    +4   +0   32  33  17   3  PPPEEEEEEEKPPKE
  0x0D0E02C6  2025-07-24 14:45:56     685    +4   +1   32  33  17   3  KPKPEKKEEKPKKEK


Elm calls (type P/E/K as heard; M = pick manually):  pee


Elm calls so far: PEE
2 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0B0E02C6  2025-07-24 14:45:54     685    +4   -1   32  33  17   3  EKEKEPKPPPEPEEE
  0x0C0E02C6  2025-07-24 14:45:55     685    +4   +0   32  33  17   3  PPPEEEEEEEKPPKE


Elm calls (type P/E/K as heard; M = pick manually):  e


Elm calls so far: PEEE
2 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0B0E02C6  2025-07-24 14:45:54     685    +4   -1   32  33  17   3  EKEKEPKPPPEPEEE
  0x0C0E02C6  2025-07-24 14:45:55     685    +4   +0   32  33  17   3  PPPEEEEEEEKPPKE


Elm calls (type P/E/K as heard; M = pick manually):  e


Elm calls so far: PEEEE
1 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0C0E02C6  2025-07-24 14:45:55     685    +4   +0   32  33  17   3  PPPEEEEEEEKPPKE

=== Seed identified: 0x0C0E02C6  2025-07-24 14:45:55  delay=685  R/E/L=32/33/17  Elm=PPPEEEEEEEKPPKE ===


## Section B — Expedition-style Seed Identification  (→ `b_seed`)

Ported from the Metronome Compass Testing notebook.  Generate every candidate seed near a
target `(time, delay)`, each with its precomputed Metronome battle path, then walk the real
battle turn by turn — candidates whose path diverges from what you observe drop out until a
single seed remains, saved as **`b_seed`** (the whole row).

Config uses `b_`-prefixed names so it won't clash with Section A.  The Metronome user's
movepool besides Metronome is `DEFAULT_EXTRA_MOVES` in `utils/calibration_tools.py`; edit it
there if it changes.  (Seeds here use the same year-correct `seed_for` as Section A.)

Magikarp's **level and gender are asked at run time** (they change each battle); the Metronome user's own gender is the stable `b_metronome_user_is_female` config.


In [95]:
# --- Section B: Metronome-compass target ---
# b_target_time     = dt.datetime(2025, 7, 24, 14, 49, 0)
b_target_time = a_target_time + dt.timedelta(seconds=200)
b_target_delay    = 12083
b_seconds_window  = 2         # +/- X seconds
b_delay_window    = 2000       # +/- Y delays
b_metronome_only  = False     # True = Metronome-only user; False = + DEFAULT_EXTRA_MOVES
b_metronome_user_is_female = True   # the Metronome user's gender (rarely changes)

# Magikarp's level + gender change every run, so prompt for them at execution time.
# opposite_gender is derived relative to the Metronome user's gender above.
b_magikarp_level, b_opposite_gender = prompt_magikarp(b_metronome_user_is_female)

b_candidates = generate_candidates_near(
    b_target_time, b_target_delay, b_seconds_window, b_delay_window,
    magikarp_level=b_magikarp_level, opposite_gender=b_opposite_gender,
    metronome_only=b_metronome_only,
)

# Walk the real battle turn by turn; candidates diverging from what you observe drop out.
b_seed = narrow_candidates(b_candidates, b_magikarp_level, b_opposite_gender,
                           metronome_only=b_metronome_only)


Magikarp level:  6
Magikarp gender (M/F):  F



20005 / 20005 seeds remain -- next is turn 1
        Seed   Delay    dD  predicted turn 1
  0xE80E2F4C   12083    +0  KspM319h           (Metal Sound)
  0xE70E2F4C   12083    +0  KspM024h           (Double Kick)
  0xE90E2F4C   12083    +0  KspM147h           (Spore)
  0xE60E2F4C   12083    +0  KspM196h~          (Icy Wind)
  0xEA0E2F4C   12083    +0  KspM286            (Imprison)
  0xE80E2F4B   12082    -1  KspM167hhh         (Triple Kick)
  0xE80E2F4D   12084    +1  KspM005h           (Mega Punch)
  0xE70E2F4B   12082    -1  KspM028h           (Sand Attack)
  0xE70E2F4D   12084    +1  KspM177h           (Aeroblast)
  0xE90E2F4B   12082    -1  KspM462h           (Crush Grip)
  0xE90E2F4D   12084    +1  KspM144?_          (Transform)
  0xE60E2F4B   12082    -1  KspM200h           (Outrage)
  0xE60E2F4D   12084    +1  KspM349            (Dragon Dance)
  0xEA0E2F4B   12082    -1  KspM290h~          (Secret Power)
  0xEA0E2F4D   12084    +1  KspM439h           (Rock Wrecker)
  ... and 199

  Metronome selected? (move name or M###):  Magnet Rise



50 / 20005 seeds remain -- next is turn 2
        Seed   Delay    dD  predicted turn 2
  0xEA0E2F5B   12098   +15  KspM332h           (Aerial Ace)
  0xE80E2F65   12108   +25  KspM421h           (Shadow Claw)
  0xE70E2F0B   12018   -65  KspM441h~          (Gunk Shot)
  0xEA0E2E0F   11766  -317  KspM005-           (Mega Punch)
  0xEA0E30A7   12430  +347  KspM349            (Dragon Dance)
  0xE80E30B1   12440  +357  KspM282h           (Knock Off)
  0xE60E30BB   12450  +367  KspM370h           (Close Combat)
  0xE70E2DBF   11686  -397  KspM113            (Light Screen)
  0xE90E2DB5   11676  -407  KspM024h           (Double Kick)
  0xE60E2D36   11549  -534  KspM034h           (Body Slam)
  0xEA0E3194   12667  +584  KspM341h~          (Mud Shot)
  0xE70E319B   12674  +591  KspM347            (Calm Mind)
  0xEA0E31F3   12762  +679  KspM209h~          (Spark)
  0xE80E31FD   12772  +689  KspM298h           (Teeter Dance)
  0xE60E3207   12782  +699  KspM387            (Last Resort)
  ... and 35

  Metronome selected? (move name or M###):  Shadow Claw
  Hit, crit, or miss? (h/!/-):  h



1 / 20005 seeds remain -- next is turn 3
        Seed   Delay    dD  predicted turn 3
  0xE80E2F65   12108   +25  KspM249h           (Rock Smash)

Seed identified: 0xE80E2F65  time=2025-07-24 14:49:15  delay=12108  dD=+25
Full path: KspM393 KspM421h KspM249h KspM438h KspM158h KspM013 Ksph KspM174 KspM287 KspM210h
Remaining Metronome moves (turn 3+):
  Turn 3: Rock Smash (M249)
  Turn 4: Power Whip (M438)
  Turn 5: Hyper Fang (M158)
  Turn 6: Razor Wind (M013)
  Turn 8: Curse (M174)
  Turn 9: Refresh (M287)
  Turn 10: Fury Cutter (M210)


## Section C — Save the run  (→ `data/compass_runs.jsonl`)

Records this calibration run — both identified seeds (`a_seed`, `b_seed`) plus the metadata
below — as one JSON line appended to `data/compass_runs.jsonl`.

You're prompted for a **run tag** (e.g. `300s Samwise`, `11000d Work`), the **target timer
delay**, the **target timer calibration**, and free-form **notes**.  Leaving the tag / delay
/ calibration blank re-uses the previous run's value (notes never default).  The record is
pretty-printed and confirmed (`y`/`n`) before it's written.


In [96]:
# --- Section C: append this run to data/compass_runs.jsonl ---
run_record = save_compass_run(a_seed, b_seed)


Run tag [200k delay]:  
Target timer delay [200000]:  
Target timer calibration [-5000]:  
Notes:  



{
  "saved_at": "2026-09-07T17:09:48",
  "tag": "200k delay",
  "target_timer_delay": 200000,
  "target_timer_calibration": -5000,
  "notes": "",
  "a_seed": {
    "seed": 202244806,
    "seed_hex": "0x0C0E02C6",
    "time": "2025-07-24T14:45:55",
    "delay": 685,
    "sec_delta": 0,
    "delay_delta": 4,
    "r_route": 32,
    "e_route": 33,
    "l_route": 17,
    "rng_calls": 3,
    "elm": "PPPEEEEEEEKPPKE"
  },
  "b_seed": {
    "seed": 3893243749,
    "seed_hex": "0xE80E2F65",
    "time": "2025-07-24T14:49:15",
    "delay": 12108,
    "sec_delta": 0,
    "delay_delta": 25,
    "path_str": "KspM393 KspM421h KspM249h KspM438h KspM158h KspM013 Ksph KspM174 KspM287 KspM210h"
  }
}



Save this run? (y/n):  y


Saved to data/compass_runs.jsonl


## Section D — Timer calibration math  (delay/calibration ↔ seed_b frame)

Fits the collected runs to answer: **given a timer countdown, what `F_b` frame will I hit?**
`M = target_timer_delay + target_timer_calibration` (ms, calibration signed).

**Two frame "rates" that are easy to confuse:**

- **within-run *average* rate** `= (F_b − F_a)/(T_b − T_a)` — the mean frames/sec over a run.
  This genuinely **rises with M** (≈56.8 → 58.3 Hz here): every run starts at frame ~670 in
  the slow post-boot region, and the longer the run, the more that slow start dilutes out,
  pulling the average up toward the ~60 Hz ceiling.
- **`dF_b/dM`** — the slope you actually need to turn a commanded `M` into `F_b`. This is the
  *instantaneous* rate at battle time, which by 3–7 min is already near the ceiling (~59.6 Hz)
  and barely changing. So **`F_b` vs `M` is essentially a straight line** — just with slope
  ~59.6, **not** the average rate.

**The models (previous vs new), all shown in the report for comparison:**

- **`within_rate` (previous)** — a line whose slope is the within-run *average* rate. Using
  the average rate as the `F_b`-vs-`M` slope is the bug that made residuals explode (~160
  frames) as `M` left the centroid, tanking predictions far from the calibrated delays.
- **`linear_m` (NEW, recommended ★)** — fits `F_b` directly against `M` (robust Theil–Sen).
  Correct slope → residuals drop to ~30 frames, and predictions hold across the whole range.
- **`quad_m` (NEW, experimental)** — adds an `M²` term to test the rising-rate curvature
  directly. So far it barely changes the RMS and its curvature is within noise (near the
  ceiling there's little left to bend), so it isn't used by default — worth re-checking as
  the sweep extends toward 10 min.

`model["recommended"]` (= `linear_m`) drives the top-level `predict` / `solve` /
`hit_probability`; each individual model is under `model["models"][name]`.

**Uncertainty is still split two ways:** reducible **mean-uncertainty** (shrinks with more
runs; ~0 near measured delays, grows as you extrapolate) and irreducible **physical jitter**
(`σ ≈ c·√M`) — the latter sets your real hit odds, so use a `tolerance` window that reflects
which frames are actually acceptable.


In [90]:
# --- Section D: fit the collected runs; predict the F_b frame from a timer delay ---
model = calibrate_timer()   # reads data/compass_runs.jsonl, prints the model comparison

# === Forward prediction: given a timer delay + calibration, what frame will I land on? ===
d_delay       = 200000    # target_timer_delay (ms)
d_calibration = -5000     # target_timer_calibration (ms, signed)

def predict_report(sub, label):
    p = sub["predict"](d_delay, d_calibration)   # range = expected +/- (band + 2*jitter)
    print(f"    [{label:<26}] F_b = {p['expected']:8.1f}   ~95% range "
          f"[{p['lo']:.0f}, {p['hi']:.0f}]   (jitter +/-{p['jitter']:.1f})")

print(f"\nWith delay={d_delay} cal={d_calibration}  (M = {d_delay + d_calibration} ms) "
      f"-> predicted F_b:")
predict_report(model["models"]["within_rate"], "previous within-rate")
predict_report(model["models"]["linear_m"],    "NEW F_b-vs-M line  (used)")
if "quad_m" in model["models"]:
    predict_report(model["models"]["quad_m"],  "NEW quadratic (experimental)")

# === How likely is that delay to land on a specific frame (within a +/- window)? ===
d_target_fb = round(model["predict"](d_delay, d_calibration)["expected"])  # default: the predicted frame
d_tolerance = 25          # half-window (frames) counted as a hit; 0.5 = exactly that frame
hp = model["hit_probability"](d_delay, d_calibration, d_target_fb, tolerance=d_tolerance)
print(f"\nP(land on F_b={d_target_fb} +/-{d_tolerance} at that delay) = {hp['p']*100:.1f}%  "
      f"(expected off by {hp['delta']:+.1f} frames)")

# === (optional) reverse lookup: which delay would center you on a target frame? ===
sol = model["solve"](d_target_fb, calibration=d_calibration)
print(f"\n(reverse) to center on F_b={d_target_fb} holding cal={d_calibration}: "
      f"delay = {sol['delay']:.0f} ms")


=== Timer calibration  (32 run(s), 32 timed, 1 excluded as outlier) ===

  F_b as a function of M = delay + calibration (ms)

  within-run avg rate 58.1393 +/- 0.0132 Hz (dF/dt; rises with M as the slow post-boot frames dilute out)

   within-run-rate slope (previous)       slope 58.1393 Hz                   RMS residual  300.7 frames
  *F_b-vs-M line (NEW)                    slope 60.0292 Hz                   RMS residual   76.4 frames
   F_b-vs-M quadratic (NEW, experimental) inst rate 59.597->60.918 Hz over M range RMS residual   64.7 frames

  ( * = recommended; predict/solve/hit_probability use it )

  tag                     M      Fa      Fb      dF    dt     rate
  11000d Smeagol     185000     673   11469   10796   190   56.821
  11000d Smeagol     180000     651   11166   10515   185   56.838
  11000d Smeagol     176913     677   11013   10336   182   56.791
  11000d Smeagol     190000     681   11417   10736   195   55.056   <- outlier (off-trend), excluded
  Sweep Smeagol  